# Extraction and Exploratory Data Analysis

**Goal:** extract and explore a real retail dataset.

**Data:** Online Retail, UCI Machine Learning Repository. UK online gift shop, 01 Dec 2010 to 09 Dec 2011.

**Why:** the shop needs to know its best products, customers and countries, but the raw data has to be checked first.

```text
Raw Data
↓
Extraction
↓
Exploratory Data Analysis
```

The raw data is only read and described in this section.

## Section 2: Imports

In [ ]:
import hashlib
import os
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

pio.renderers.default = "vscode"

print("pandas", pd.__version__)

## Section 3: Locate the Dataset

Looks for the zip next to the notebook, in a `data` folder, or in Downloads. The zip is only read. The extracted copy goes to a temp folder.

In [ ]:
WORK_DIR = Path(tempfile.gettempdir()) / "online_retail_capstone"
RAW_DIR = WORK_DIR / "raw"
WORK_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)


def find_project_dir():
    here = Path.cwd()
    for folder in [here, here / "notebooks", here.parent]:
        if (folder / "01_extraction_and_eda.ipynb").exists():
            return folder
    return here


PROJECT_DIR = find_project_dir()


def find_zip():
    base = PROJECT_DIR
    folders = [base, base / "data", base.parent, base.parent / "data", Path.cwd(), Path.cwd() / "data", Path.home() / "Downloads"]
    for folder in folders:
        if folder.is_dir():
            matches = sorted(folder.glob("online*retail*.zip"))
            if matches:
                return matches[0]
    raise FileNotFoundError("online+retail.zip was not found next to the notebooks, in a data folder or in Downloads")


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
zip_path = find_zip()
zip_hash = file_sha256(zip_path)
print("ZIP file     :", zip_path.name)
print("Size (MB)    :", round(zip_path.stat().st_size / 1024 ** 2, 2))
print("SHA-256      :", zip_hash)

In [ ]:
with zipfile.ZipFile(zip_path) as archive:
    archive_contents = pd.DataFrame(
        [{"file": item.filename, "size_mb": round(item.file_size / 1024 ** 2, 2)} for item in archive.infolist()]
    )
archive_contents

In [ ]:
with zipfile.ZipFile(zip_path) as archive:
    excel_members = [name for name in archive.namelist() if name.lower().endswith((".xlsx", ".xls"))]
    if not excel_members:
        raise FileNotFoundError("No Excel file found inside the ZIP archive")
    archive.extract(excel_members[0], RAW_DIR)

excel_path = RAW_DIR / excel_members[0]
print("Excel dataset:", excel_path.name)
print("Exists       :", excel_path.exists())
print("ZIP unchanged:", file_sha256(zip_path) == zip_hash)

## Section 4: Extraction

Load the Excel file into `raw_df`. Takes about a minute.

In [ ]:
raw_df = pd.read_excel(excel_path)
print("Rows   :", f"{raw_df.shape[0]:,}")
print("Columns:", raw_df.shape[1])

In [ ]:
pd.DataFrame({"position": range(1, len(raw_df.columns) + 1), "column": raw_df.columns})

In [ ]:
raw_df.head(10)

In [ ]:
raw_df.tail(10)

In [ ]:
raw_df.dtypes.rename("dtype").to_frame()

## Section 5: Basic EDA

In [ ]:
raw_df.shape

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe()

In [ ]:
raw_df[["InvoiceNo", "StockCode", "Description", "Country"]].astype(str).describe()

In [ ]:
unique_values = raw_df.nunique(dropna=True).rename("unique_values").to_frame()
unique_values

In [ ]:
invoice_text = raw_df["InvoiceNo"].astype(str)
stock_text = raw_df["StockCode"].astype(str)

key_counts = pd.DataFrame(
    {
        "metric": ["Transaction lines", "Unique invoices", "Unique customers", "Unique products (StockCode)", "Unique countries"],
        "value": [len(raw_df), invoice_text.nunique(), raw_df["CustomerID"].nunique(), stock_text.nunique(), raw_df["Country"].nunique()],
    }
)
key_counts

## Section 6: Data Quality Investigation

Only checking for problems here. Nothing is changed.

### 6.1 Missing values

In [ ]:
missing_table = pd.DataFrame(
    {"missing_count": raw_df.isna().sum(), "missing_percent": (raw_df.isna().mean() * 100).round(2)}
)
missing_table

In [ ]:
missing_counts = raw_df.isna().sum()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(missing_counts.index, missing_counts.values, color="steelblue")
ax.set_xlabel("Missing values")
ax.set_title("Missing values per column")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 6.2 Duplicate rows and invoice structure

In [ ]:
duplicate_mask = raw_df.duplicated(keep=False)
duplicate_row_count = int(raw_df.duplicated().sum())
print("Fully duplicated rows (extra copies):", f"{duplicate_row_count:,}", f"({duplicate_row_count / len(raw_df) * 100:.2f}% of rows)")
raw_df[duplicate_mask].sort_values(["InvoiceNo", "StockCode"]).head(10)

In [ ]:
lines_per_invoice = raw_df.groupby(invoice_text).size()
invoice_customers = raw_df.groupby(invoice_text)["CustomerID"].nunique(dropna=False)
invoice_timestamps = raw_df.groupby(invoice_text)["InvoiceDate"].nunique()

invoice_structure = pd.DataFrame(
    {
        "check": [
            "Invoices",
            "Invoices with more than one line (expected, an invoice holds many products)",
            "Invoices linked to more than one CustomerID",
            "Invoices with more than one InvoiceDate value",
        ],
        "count": [len(lines_per_invoice), int((lines_per_invoice > 1).sum()), int((invoice_customers > 1).sum()), int((invoice_timestamps > 1).sum())],
    }
)
invoice_structure

In [ ]:
lines_per_invoice.describe().to_frame("lines_per_invoice")

### 6.3 Quantity and price

In [ ]:
quantity_issues = pd.DataFrame(
    {
        "check": ["Quantity equal to zero", "Quantity below zero", "UnitPrice equal to zero", "UnitPrice below zero"],
        "rows": [
            int((raw_df["Quantity"] == 0).sum()),
            int((raw_df["Quantity"] < 0).sum()),
            int((raw_df["UnitPrice"] == 0).sum()),
            int((raw_df["UnitPrice"] < 0).sum()),
        ],
    }
)
quantity_issues["percent_of_rows"] = (quantity_issues["rows"] / len(raw_df) * 100).round(3)
quantity_issues

In [ ]:
raw_df[raw_df["Quantity"] < 0].head(8)

In [ ]:
raw_df[raw_df["UnitPrice"] <= 0].head(8)

In [ ]:
extreme_rows = raw_df[raw_df["Quantity"].abs() >= 10000]
print("Rows with absolute quantity of 10,000 or more:", len(extreme_rows))
extreme_rows

### 6.4 Missing CustomerID and missing descriptions

In [ ]:
missing_customer = raw_df["CustomerID"].isna()
missing_description = raw_df["Description"].isna()
raw_revenue_line = raw_df["Quantity"] * raw_df["UnitPrice"]

customer_issue_table = pd.DataFrame(
    {
        "check": [
            "Rows without CustomerID",
            "Rows without Description",
            "Rows missing both",
            "Rows without Description that have a CustomerID",
            "Unit price zero among rows without Description",
        ],
        "rows": [
            int(missing_customer.sum()),
            int(missing_description.sum()),
            int((missing_customer & missing_description).sum()),
            int((~missing_customer & missing_description).sum()),
            int((missing_description & (raw_df["UnitPrice"] == 0)).sum()),
        ],
    }
)
customer_issue_table

In [ ]:
anonymous_revenue = raw_revenue_line[missing_customer].sum()
total_raw_revenue = raw_revenue_line.sum()
print("Quantity x UnitPrice on rows without CustomerID:", f"{anonymous_revenue:,.2f}")
print("Share of the raw revenue total                 :", f"{anonymous_revenue / total_raw_revenue * 100:.1f}%")

### 6.5 Cancelled invoices

In [ ]:
cancelled_mask = invoice_text.str.startswith("C")
adjustment_mask = invoice_text.str.startswith("A")
negative_not_cancelled = (raw_df["Quantity"] < 0) & ~cancelled_mask

cancel_table = pd.DataFrame(
    {
        "check": [
            "Lines on invoices starting with C (cancellations)",
            "Distinct cancelled invoices",
            "Lines on invoices starting with A",
            "Negative quantity lines that are not cancellations",
            "Cancelled lines with a positive quantity",
        ],
        "count": [
            int(cancelled_mask.sum()),
            int(invoice_text[cancelled_mask].nunique()),
            int(adjustment_mask.sum()),
            int(negative_not_cancelled.sum()),
            int((cancelled_mask & (raw_df["Quantity"] > 0)).sum()),
        ],
    }
)
cancel_table

In [ ]:
raw_df[cancelled_mask].head(6)

In [ ]:
raw_df[adjustment_mask]

### 6.6 Date format and data types

In [ ]:
date_summary = pd.DataFrame(
    {
        "check": ["dtype", "Missing dates", "Earliest invoice", "Latest invoice", "Distinct calendar days", "Rows with non-zero seconds"],
        "result": [
            str(raw_df["InvoiceDate"].dtype),
            int(raw_df["InvoiceDate"].isna().sum()),
            raw_df["InvoiceDate"].min(),
            raw_df["InvoiceDate"].max(),
            raw_df["InvoiceDate"].dt.normalize().nunique(),
            int((raw_df["InvoiceDate"].dt.second != 0).sum()),
        ],
    }
)
date_summary.astype(str)

In [ ]:
type_check = pd.DataFrame(
    {
        column: raw_df[column].map(lambda value: type(value).__name__).value_counts()
        for column in ["InvoiceNo", "StockCode", "CustomerID"]
    }
).fillna(0).astype(int)
type_check

### 6.7 Unexpected values

In [ ]:
product_pattern = stock_text.str.match(r"^\d{5}")
non_product_codes = (
    raw_df[~product_pattern]
    .assign(StockCode=stock_text[~product_pattern])
    .groupby("StockCode")
    .agg(lines=("StockCode", "size"), example_description=("Description", "first"))
    .sort_values("lines", ascending=False)
)
print("Distinct stock codes that do not start with a 5-digit product number:", len(non_product_codes))
non_product_codes.head(15)

In [ ]:
descriptions_per_code = raw_df.groupby(stock_text)["Description"].nunique()
prices_per_code = raw_df.groupby(stock_text)["UnitPrice"].nunique()
countries_per_customer = raw_df.groupby("CustomerID")["Country"].nunique()
description_text = raw_df["Description"].dropna().astype(str)

unexpected_table = pd.DataFrame(
    {
        "check": [
            "StockCodes with more than one description",
            "StockCodes with more than one UnitPrice",
            "Customers appearing under more than one country",
            "Descriptions with leading or trailing spaces",
            "Descriptions that contain lowercase letters",
        ],
        "count": [
            int((descriptions_per_code > 1).sum()),
            int((prices_per_code > 1).sum()),
            int((countries_per_customer > 1).sum()),
            int((description_text != description_text.str.strip()).sum()),
            int(description_text.str.contains(r"[a-z]").sum()),
        ],
    }
)
unexpected_table

In [ ]:
raw_df["Country"].value_counts().tail(8).to_frame("lines")

### 6.8 Data quality summary

All problem counts in one table.

In [ ]:
quality_summary = pd.DataFrame(
    [
        ("Rows without CustomerID", int(missing_customer.sum())),
        ("Rows without Description", int(missing_description.sum())),
        ("Fully duplicated rows", duplicate_row_count),
        ("Lines on cancelled invoices (prefix C)", int(cancelled_mask.sum())),
        ("Lines with Quantity below zero", int((raw_df["Quantity"] < 0).sum())),
        ("Lines with UnitPrice equal to zero", int((raw_df["UnitPrice"] == 0).sum())),
        ("Lines with UnitPrice below zero", int((raw_df["UnitPrice"] < 0).sum())),
        ("Lines with a non-product StockCode", int((~product_pattern).sum())),
    ],
    columns=["issue", "rows"],
)
quality_summary["percent_of_rows"] = (quality_summary["rows"] / len(raw_df) * 100).round(2)
quality_summary.sort_values("rows", ascending=False).reset_index(drop=True)

## Section 7: Initial Exploratory Analysis

Charts use the raw data. Rankings only use positive quantities. Revenue = Quantity x UnitPrice, not saved in `raw_df`.

In [ ]:
transactions_by_country = (
    raw_df.assign(invoice=invoice_text)
    .groupby("Country")
    .agg(invoices=("invoice", "nunique"), lines=("invoice", "size"))
    .sort_values("invoices", ascending=False)
)
transactions_by_country.head(10)

In [ ]:
fig = px.bar(
    transactions_by_country.head(10).reset_index(),
    x="Country",
    y="invoices",
    title="Top 10 countries by number of invoices",
    labels={"invoices": "Invoices"},
)
fig.show()

In [ ]:
customers_by_country = (
    raw_df.dropna(subset=["CustomerID"])
    .groupby("Country")["CustomerID"]
    .nunique()
    .sort_values(ascending=False)
    .rename("customers")
)
fig = px.bar(
    customers_by_country.head(10).reset_index(),
    x="Country",
    y="customers",
    title="Top 10 countries by number of identified customers",
)
fig.show()

In [ ]:
positive_lines = raw_df[raw_df["Quantity"] > 0].assign(StockCode=stock_text)

top_products_quantity = (
    positive_lines.groupby("StockCode")
    .agg(description=("Description", "first"), total_quantity=("Quantity", "sum"))
    .sort_values("total_quantity", ascending=False)
    .head(10)
    .reset_index()
)
top_products_quantity

In [ ]:
fig = px.bar(
    top_products_quantity.sort_values("total_quantity"),
    x="total_quantity",
    y="description",
    orientation="h",
    title="Top 10 products by quantity sold (raw data)",
)
fig.show()

In [ ]:
top_products_frequency = (
    positive_lines.assign(invoice=invoice_text)
    .groupby("StockCode")
    .agg(description=("Description", "first"), invoices_containing_product=("invoice", "nunique"))
    .sort_values("invoices_containing_product", ascending=False)
    .head(10)
    .reset_index()
)
top_products_frequency

In [ ]:
revenue_overview = pd.DataFrame(
    {
        "measure": [
            "Sum of Quantity x UnitPrice, all rows",
            "Sum for rows with positive quantity",
            "Sum for rows with negative quantity",
            "Sum for rows without CustomerID",
        ],
        "value": [
            raw_revenue_line.sum(),
            raw_revenue_line[raw_df["Quantity"] > 0].sum(),
            raw_revenue_line[raw_df["Quantity"] < 0].sum(),
            raw_revenue_line[missing_customer].sum(),
        ],
    }
)
revenue_overview["value"] = revenue_overview["value"].map("{:,.2f}".format)
revenue_overview

In [ ]:
monthly_volume = (
    raw_df.assign(month=raw_df["InvoiceDate"].dt.to_period("M").astype(str), invoice=invoice_text)
    .groupby("month")
    .agg(invoices=("invoice", "nunique"), lines=("invoice", "size"))
    .reset_index()
)
monthly_volume

In [ ]:
fig = px.line(
    monthly_volume,
    x="month",
    y="invoices",
    markers=True,
    title="Monthly transaction volume (unique invoices, raw data)",
)
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(raw_df.loc[raw_df["Quantity"].between(-50, 100), "Quantity"], bins=75, color="steelblue")
ax.set_xlabel("Quantity (clipped to the range -50 to 100 for display)")
ax.set_ylabel("Lines")
ax.set_title("Distribution of Quantity in the raw data")
plt.tight_layout()
plt.show()

## Section 8: Initial Findings

Numbers below come from the code above.

In [ ]:
from IPython.display import Markdown, display

top_country = transactions_by_country.index[0]
top_country_share = transactions_by_country["invoices"].iloc[0] / transactions_by_country["invoices"].sum() * 100
busiest_month = monthly_volume.loc[monthly_volume["invoices"].idxmax()]
complete_months = monthly_volume.iloc[:-1]
quietest_month = complete_months.loc[complete_months["invoices"].idxmin()]
partial_month = monthly_volume.iloc[-1]

findings = f"""
**Dataset obtained.** `{excel_path.name}` was extracted from `{zip_path.name}` and loaded into `raw_df` with {len(raw_df):,} rows and {raw_df.shape[1]} columns, covering {raw_df['InvoiceDate'].min():%d %B %Y} to {raw_df['InvoiceDate'].max():%d %B %Y}.

**Scale.** The data holds {invoice_text.nunique():,} invoices, {raw_df['CustomerID'].nunique():,} identified customers, {stock_text.nunique():,} stock codes and {raw_df['Country'].nunique()} countries.

**Data quality problems discovered**

- {int(missing_customer.sum()):,} rows ({missing_customer.mean() * 100:.1f}%) have no CustomerID, and {int(missing_description.sum()):,} rows have no Description.
- {duplicate_row_count:,} rows are exact duplicates of another row.
- {int(cancelled_mask.sum()):,} lines belong to cancelled invoices, and {int((raw_df['Quantity'] < 0).sum()):,} lines in total have a negative quantity.
- {int((raw_df['UnitPrice'] <= 0).sum()):,} lines have a zero or negative unit price.
- {int((~product_pattern).sum()):,} lines use stock codes that are not product numbers, such as postage and manual adjustments.
- InvoiceNo and StockCode contain a mix of numbers and text, and {int((descriptions_per_code > 1).sum()):,} stock codes have more than one description.

**Patterns discovered**

- {top_country} accounts for {top_country_share:.1f}% of all invoices.
- Monthly invoice volume is highest in {busiest_month['month']} ({int(busiest_month['invoices']):,} invoices) and lowest among complete months in {quietest_month['month']} ({int(quietest_month['invoices']):,} invoices). The final month, {partial_month['month']}, is incomplete because the data ends on {raw_df['InvoiceDate'].max():%d %B %Y}.
- Quantity contains extreme positive and negative values that need inspection before any analysis.
"""
display(Markdown(findings))